In [3]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
sys.path.insert(0, str(PROJECT_ROOT))

In [4]:
import os
import pandas as pd

# =========================
# Configuration
# =========================
base_path = "cv_optuna_utcnn/test_best_random_on_spCV"
base_path2 = "../Results/evaluation_on_bestconfigs"

model = "UTCNN"

# Default values
DEFAULT_NORMALIZATION = "StandardScaler"
DEFAULT_DEPTH = "all_depth"

# Output columns and order
OUTPUT_COLUMNS = [
    "soil_property",
    "cv_method",
    "normalization",
    "depth",
    "rmse",
    "mse",
    "r2",
    "mae",
    "time"
]


# =========================
# Process experiment files
# =========================
for region in os.listdir(base_path):

    region_path = os.path.join(base_path, region)

    if not os.path.isdir(region_path):
        continue

    # Collect data separately for each CV method
    for cv_method in os.listdir(region_path):

        cv_path = os.path.join(region_path, cv_method)

        if not os.path.isdir(cv_path):
            continue

        results = []

        # Each directory should be named Output[soil_prop]
        for output_dir in os.listdir(cv_path):

            output_path = os.path.join(cv_path, output_dir)

            if not os.path.isdir(output_path):
                continue

            metrics_file = os.path.join(
                output_path,
                "metrics_overall.csv"
            )

            if not os.path.isfile(metrics_file):
                print(f"Skipping missing file: {metrics_file}")
                continue

            # Extract soil property from directory name
            # Example: OutputOM -> OM
            soil_property = (
                output_dir[len("output"):]
                if output_dir.startswith("output")
                else output_dir
            )

            try:
                df = pd.read_csv(metrics_file)

                if df.empty:
                    print(f"Skipping empty file: {metrics_file}")
                    continue

                # Use the first row of overall metrics
                metrics = df.iloc[0]

                # Case-insensitive column matching
                column_map = {
                    col.strip().lower(): col
                    for col in df.columns
                }

                def get_metric(name, default=None):
                    col = column_map.get(name.lower())
                    return metrics[col] if col is not None else default

                results.append({
                    "soil_property": soil_property,
                    "cv_method": cv_method,
                    "normalization": DEFAULT_NORMALIZATION,
                    "depth": DEFAULT_DEPTH,
                    "rmse": get_metric("RMSE"),
                    "mse": get_metric("MSE"),
                    "r2": get_metric("R2"),
                    "mae": None,
                    "time": None
                })

            except Exception as e:
                print(f"Error processing {metrics_file}: {e}")

        # Save one combined CSV for this region and CV method
        if results:
            output_df = pd.DataFrame(
                results,
                columns=OUTPUT_COLUMNS
            )

            output_dir = os.path.join(
                base_path2,
                region,
                 model
            )
            os.makedirs(output_dir, exist_ok=True)

            output_file = os.path.join(
                output_dir,
                f"experiments_{model}_{cv_method}_2d_dm_new.csv"
            )

            output_df.to_csv(output_file, index=False)

            print(
                f"Saved {len(output_df)} properties to: "
                f"{output_file}"
            )

print("Processing completed.")

Saved 7 properties to: ../Results/evaluation_on_bestconfigs\all\UTCNN\experiments_UTCNN_block_k5_10000_2d_dm_new.csv
Saved 7 properties to: ../Results/evaluation_on_bestconfigs\all\UTCNN\experiments_UTCNN_block_k5_20000_2d_dm_new.csv
Saved 7 properties to: ../Results/evaluation_on_bestconfigs\all\UTCNN\experiments_UTCNN_block_k5_30000_2d_dm_new.csv
Saved 7 properties to: ../Results/evaluation_on_bestconfigs\all\UTCNN\experiments_UTCNN_block_k5_50000_2d_dm_new.csv
Saved 7 properties to: ../Results/evaluation_on_bestconfigs\all\UTCNN\experiments_UTCNN_random_2d_dm_new.csv
Saved 7 properties to: ../Results/evaluation_on_bestconfigs\all\UTCNN\experiments_UTCNN_sp1_k5_10000_2d_dm_new.csv
Saved 7 properties to: ../Results/evaluation_on_bestconfigs\all\UTCNN\experiments_UTCNN_sp1_k5_20000_2d_dm_new.csv
Saved 7 properties to: ../Results/evaluation_on_bestconfigs\all\UTCNN\experiments_UTCNN_sp1_k5_30000_2d_dm_new.csv
Saved 7 properties to: ../Results/evaluation_on_bestconfigs\all\UTCNN\experime